# General setup

In [1]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [2]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [3]:
# Setup
model_path = './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/' # './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/'  './models/Interpolation_v2_Unitcell_latentLog_2d_latentMSE_biggerDecoder/'
model_name = 'best_model_annealed.pth' # 'best_model_annealed.pth'    'best_model.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# latent_space_version = '_prior' # '' or '_prior'

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
legend_order = [
    'RheniumTrioxide', 
    'Wurtzite', 
    'CaesiumChloride', 
    'RockSalt', 
    'NickelArsenide', 
    'Rutile', 
    'ZincBlende', 
    'CadmiumIodide',
    'AntiFluorite',
    'Fluorite',
    'CadmiumChloride',
    'Spinel',
]

In [5]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [6]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [7]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

# Change column names
df_rec.rename(columns={
    'cell_parameters': 'pred_cell_parameters',
    'cell_atoms': 'pred_cell_atoms',
    'cell_positions': 'pred_cell_positions',
    'cell_parameters_prior': 'pred_cell_parameters_prior',
    'cell_atoms_prior': 'pred_cell_atoms_prior',
    'cell_positions_prior': 'pred_cell_positions_prior',
}, inplace=True)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
    df_rec[['ls1_prior', 'ls2_prior']] = df_rec['latent_space_mean_prior'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_rec['latent_space_mean_prior'].values.tolist()))

In [8]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-5.160841,0.005682,0.004352,0.001329,4.257474e-09,0.000049,Rutile,56.331192
1,-7.088384,0.000687,0.000360,0.000326,9.515396e-07,0.000143,RheniumTrioxide,11.699970
2,-7.935185,0.000290,0.000101,0.000188,1.096294e-06,0.000065,RheniumTrioxide,53.608574
3,-5.964068,0.001043,0.000640,0.000403,2.767358e-08,0.001467,Spinel,22.515327
4,-5.911193,0.001250,0.000766,0.000484,1.277242e-08,0.001393,Spinel,43.861179


In [9]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,pred_cell_parameters,pred_cell_positions,pred_cell_atoms,pred_cell_parameters_prior,pred_cell_positions_prior,pred_cell_atoms_prior,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,Rutile,56,32,24,"[4.540275573730469, 4.525632381439209, 2.85991...","[[0.016720000654459, -0.0175199992954731, 0.00...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[4.5490593910217285, 4.538936614990234, 2.8607...","[[0.007027522660791874, -0.021485134959220886,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",...,"[0.08507634699344635, 0.1256801038980484]","[0.33626455068588257, -1.4690507650375366]","[0.0853525847196579, 0.12606032192707062]","[0.17788006365299225, 0.17788006365299225, -0....","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[37, 8, 37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 3...",0.336592,-1.465637,0.336265,-1.469051
1,RheniumTrioxide,56,36,20,"[3.8748703002929688, 3.8799965381622314, 3.769...","[[0.0010499999625608325, 0.00431999983265996, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[3.8880534172058105, 3.8946597576141357, 3.768...","[[0.002416004426777363, 0.004546537064015865, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",...,"[0.0597555972635746, 0.08878665417432785]","[-0.3214685320854187, 0.898919403553009]","[0.061083149164915085, 0.09021936357021332]","[-0.31774070858955383, -0.31774070858955383, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[49, 8, 8, 8, 49, 49, 49, 8, 8, 8, 8, 8, 8, 49...",-0.320329,0.900893,-0.321469,0.898919
2,RheniumTrioxide,56,36,20,"[3.8752238750457764, 3.8808555603027344, 3.861...","[[0.0006000000284984708, 0.00865000020712614, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[3.9153122901916504, 3.9254205226898193, 4.133...","[[0.02075561136007309, 0.025669559836387634, 0...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",...,"[0.06127431243658066, 0.09047353267669678]","[-0.32346194982528687, 0.9044766426086426]","[0.06192621961236, 0.09163868427276611]","[-0.31806865334510803, -0.31806865334510803, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[52, 8, 8, 8, 52, 52, 52, 8, 8, 8, 8, 8, 8, 52...",-0.323451,0.906502,-0.323462,0.904477
3,Spinel,56,32,24,"[8.372520446777344, 8.366039276123047, 8.35763...","[[0.5196899771690369, 0.5156599879264832, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[8.463068008422852, 8.457047462463379, 8.43695...","[[0.5151798725128174, 0.513577938079834, 0.499...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",...,"[0.05441682040691376, 0.12149783223867416]","[-0.8512399792671204, 0.17954900860786438]","[0.055175475776195526, 0.12234848737716675]","[2.435340166091919, 2.435340166091919, 0.83507...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 4...",-0.843638,0.187292,-0.851240,0.179549
4,Spinel,56,32,24,"[8.673946380615234, 8.670553207397461, 8.63094...","[[0.5145999789237976, 0.5127999782562256, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[8.917573928833008, 8.919380187988281, 8.90549...","[[0.5078892707824707, 0.5087736248970032, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",...,"[0.05733315646648407, 0.12760566174983978]","[-0.8958567976951599, 0.15143229067325592]","[0.057946451008319855, 0.128786101937294]","[2.534827709197998, 2.534827709197998, 0.88602...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 7...",-0.888164,0.159848,-0.895857,0.151432


In [10]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-5.160841,0.005682,0.004352,0.001329,4.257474e-09,0.000049,56.331192,Rutile,56,32,...,"[0.08507634699344635, 0.1256801038980484]","[0.33626455068588257, -1.4690507650375366]","[0.0853525847196579, 0.12606032192707062]","[0.17788006365299225, 0.17788006365299225, -0....","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[37, 8, 37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 3...",0.336592,-1.465637,0.336265,-1.469051
1,-7.088384,0.000687,0.000360,0.000326,9.515396e-07,0.000143,11.699970,RheniumTrioxide,56,36,...,"[0.0597555972635746, 0.08878665417432785]","[-0.3214685320854187, 0.898919403553009]","[0.061083149164915085, 0.09021936357021332]","[-0.31774070858955383, -0.31774070858955383, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[49, 8, 8, 8, 49, 49, 49, 8, 8, 8, 8, 8, 8, 49...",-0.320329,0.900893,-0.321469,0.898919
2,-7.935185,0.000290,0.000101,0.000188,1.096294e-06,0.000065,53.608574,RheniumTrioxide,56,36,...,"[0.06127431243658066, 0.09047353267669678]","[-0.32346194982528687, 0.9044766426086426]","[0.06192621961236, 0.09163868427276611]","[-0.31806865334510803, -0.31806865334510803, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[52, 8, 8, 8, 52, 52, 52, 8, 8, 8, 8, 8, 8, 52...",-0.323451,0.906502,-0.323462,0.904477
3,-5.964068,0.001043,0.000640,0.000403,2.767358e-08,0.001467,22.515327,Spinel,56,32,...,"[0.05441682040691376, 0.12149783223867416]","[-0.8512399792671204, 0.17954900860786438]","[0.055175475776195526, 0.12234848737716675]","[2.435340166091919, 2.435340166091919, 0.83507...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 4...",-0.843638,0.187292,-0.851240,0.179549
4,-5.911193,0.001250,0.000766,0.000484,1.277242e-08,0.001393,43.861179,Spinel,56,32,...,"[0.05733315646648407, 0.12760566174983978]","[-0.8958567976951599, 0.15143229067325592]","[0.057946451008319855, 0.128786101937294]","[2.534827709197998, 2.534827709197998, 0.88602...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 7...",-0.888164,0.159848,-0.895857,0.151432


In [11]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

Index(['epoch', 'posterior_mean', 'posterior_std', 'prior_mean', 'prior_std',
       'np_size', 'crystal_type', 'crystal_system', 'space_group', 'ls1',
       'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')


## Loss statistics

In [12]:
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-5.160841,0.005682,0.004352,0.001329,4.257474e-09,0.000049,56.331192,Rutile,56,32,...,"[0.08507634699344635, 0.1256801038980484]","[0.33626455068588257, -1.4690507650375366]","[0.0853525847196579, 0.12606032192707062]","[0.17788006365299225, 0.17788006365299225, -0....","[[0.0, 0.0, 0.0], [0.30000001192092896, 0.3000...","[37, 8, 37, 8, 8, 8, 37, 37, 37, 8, 8, 8, 8, 3...",0.336592,-1.465637,0.336265,-1.469051
1,-7.088384,0.000687,0.000360,0.000326,9.515396e-07,0.000143,11.699970,RheniumTrioxide,56,36,...,"[0.0597555972635746, 0.08878665417432785]","[-0.3214685320854187, 0.898919403553009]","[0.061083149164915085, 0.09021936357021332]","[-0.31774070858955383, -0.31774070858955383, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[49, 8, 8, 8, 49, 49, 49, 8, 8, 8, 8, 8, 8, 49...",-0.320329,0.900893,-0.321469,0.898919
2,-7.935185,0.000290,0.000101,0.000188,1.096294e-06,0.000065,53.608574,RheniumTrioxide,56,36,...,"[0.06127431243658066, 0.09047353267669678]","[-0.32346194982528687, 0.9044766426086426]","[0.06192621961236, 0.09163868427276611]","[-0.31806865334510803, -0.31806865334510803, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[52, 8, 8, 8, 52, 52, 52, 8, 8, 8, 8, 8, 8, 52...",-0.323451,0.906502,-0.323462,0.904477
3,-5.964068,0.001043,0.000640,0.000403,2.767358e-08,0.001467,22.515327,Spinel,56,32,...,"[0.05441682040691376, 0.12149783223867416]","[-0.8512399792671204, 0.17954900860786438]","[0.055175475776195526, 0.12234848737716675]","[2.435340166091919, 2.435340166091919, 0.83507...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 46, 4...",-0.843638,0.187292,-0.851240,0.179549
4,-5.911193,0.001250,0.000766,0.000484,1.277242e-08,0.001393,43.861179,Spinel,56,32,...,"[0.05733315646648407, 0.12760566174983978]","[-0.8958567976951599, 0.15143229067325592]","[0.057946451008319855, 0.128786101937294]","[2.534827709197998, 2.534827709197998, 0.88602...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 7...",-0.888164,0.159848,-0.895857,0.151432


In [13]:
df_combined.columns

Index(['total', 'reconstruction_loss', 'cell_parameters', 'cell_positions',
       'cell_atoms', 'kld', 'particleSize', 'crystalType', 'n_atoms',
       'n_oxygens', 'n_metals', 'pred_cell_parameters', 'pred_cell_positions',
       'pred_cell_atoms', 'pred_cell_parameters_prior',
       'pred_cell_positions_prior', 'pred_cell_atoms_prior',
       'latent_space_mean', 'latent_space_std', 'latent_space_mean_prior',
       'latent_space_std_prior', 'true_cell_parameters', 'true_cell_positions',
       'true_cell_atoms', 'ls1', 'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')

In [14]:
df_combined.describe()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,n_atoms,n_oxygens,n_metals,ls1,ls2,ls1_prior,ls2_prior
count,318.000000,318.000000,318.000000,318.000000,3.180000e+02,318.000000,318.000000,318.0,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000
mean,-5.442417,0.009807,0.006621,0.001882,1.303889e-03,0.015106,33.215138,56.0,27.761006,28.238994,0.130876,0.216783,0.134337,0.214255
std,1.371014,0.030809,0.013927,0.008675,2.101063e-02,0.230335,14.358678,0.0,5.986798,5.986798,0.552187,0.890453,0.545086,0.889574
min,-8.311192,0.000178,0.000040,0.000123,0.000000e+00,0.000004,9.491941,56.0,18.000000,19.000000,-1.626062,-1.483855,-1.610197,-1.488530
25%,-6.320474,0.001040,0.000594,0.000292,2.341610e-08,0.000100,22.591420,56.0,21.250000,24.000000,-0.311376,-0.417688,-0.312003,-0.412451
50%,-5.587794,0.003092,0.002047,0.000465,9.377020e-07,0.000356,33.602766,56.0,28.500000,27.500000,0.069005,0.175551,0.074068,0.175437
75%,-4.619535,0.008273,0.006295,0.000832,3.190897e-06,0.001071,43.914069,56.0,32.000000,34.750000,0.537122,0.888272,0.545362,0.869331
max,1.535497,0.479340,0.103915,0.104365,3.738800e-01,4.096343,56.331192,56.0,37.000000,38.000000,1.199861,2.154413,1.198949,2.142037


In [15]:
# Describe for each crystal type
df_combined[['crystalType', 'cell_parameters', 'cell_positions', 'cell_atoms']].groupby('crystalType').describe(percentiles=[0.5])

cell_parameters                                          \
                          count      mean       std       min       50%   
crystalType                                                               
AntiFluorite               26.0  0.012128  0.018343  0.000133  0.003626   
CadmiumChloride            27.0  0.005546  0.005974  0.000200  0.003442   
CadmiumIodide              26.0  0.004281  0.006566  0.000302  0.001504   
CaesiumChloride            27.0  0.004120  0.007744  0.000649  0.001612   
Fluorite                   27.0  0.006951  0.009789  0.000074  0.004081   
NickelArsenide             26.0  0.007181  0.017523  0.000050  0.002144   
RheniumTrioxide            27.0  0.000648  0.001024  0.000040  0.000261   
RockSalt                   27.0  0.014149  0.023988  0.000096  0.005402   
Rutile                     26.0  0.004107  0.005280  0.000746  0.002482   
Spinel                     26.0  0.005773  0.007074  0.000090  0.001507   
Wurtzite                   26.0  0.003262  0.003917  0.000177  0.001311   
ZincBlende                 27.0  0.011198  0.025827  0.000169  0.002352   

                          cell_positions                                \
                      max          count      mean       std       min   
crystalType                                                              
AntiFluorite     0.057527           26.0  0.000392  0.000148  0.000165   
CadmiumChloride  0.022115           27.0  0.001353  0.003361  0.000434   
CadmiumIodide    0.027625           26.0  0.003262  0.014610  0.000175   
CaesiumChloride  0.039921           27.0  0.002523  0.004213  0.001034   
Fluorite         0.042268           27.0  0.000505  0.001087  0.000167   
NickelArsenide   0.089987           26.0  0.001181  0.002778  0.000434   
RheniumTrioxide  0.003426           27.0  0.000352  0.000312  0.000137   
RockSalt         0.091755           27.0  0.002777  0.006459  0.000123   
Rutile           0.022498           26.0  0.003792  0.014557  0.000645   
Spinel           0.022163           26.0  0.000734  0.000703  0.000351   
Wurtzite         0.016394           26.0  0.004748  0.020347  0.000261   
ZincBlende       0.103915           27.0  0.001069  0.002461  0.000193   

                                    cell_atoms                              \
                      50%       max      count          mean           std   
crystalType                                                                  
AntiFluorite     0.000363  0.000890       26.0  4.494422e-06  3.841417e-06   
CadmiumChloride  0.000565  0.018069       27.0  5.743260e-05  2.762587e-04   
CadmiumIodide    0.000220  0.074868       26.0  9.773894e-04  4.979068e-03   
CaesiumChloride  0.001712  0.023542       27.0  3.172901e-06  1.582279e-05   
Fluorite         0.000199  0.005262       27.0  1.510689e-05  7.262509e-05   
NickelArsenide   0.000591  0.014753       26.0  2.675642e-07  6.978873e-07   
RheniumTrioxide  0.000289  0.001738       27.0  4.822641e-06  1.186163e-05   
RockSalt         0.000416  0.023625       27.0  4.311671e-04  1.251266e-03   
Rutile           0.000951  0.075155       26.0  4.257466e-08  7.973623e-08   
Spinel           0.000490  0.002948       26.0  5.346393e-08  8.509238e-08   
Wurtzite         0.000400  0.104365       26.0  1.439251e-02  7.332136e-02   
ZincBlende       0.000369  0.012111       27.0  3.989981e-05  1.158605e-04   

                                                           
                          min           50%           max  
crystalType                                                
AntiFluorite     9.621851e-07  3.465518e-06  1.934437e-05  
CadmiumChloride  8.749082e-07  2.103175e-06  1.438907e-03  
CadmiumIodide    0.000000e+00  7.450580e-09  2.538927e-02  
CaesiumChloride  0.000000e+00  1.490116e-08  8.233474e-05  
Fluorite         3.405979e-08  2.469332e-07  3.783419e-04  
NickelArsenide   6.386212e-09  2.022300e-08  2.888668e-06  
RheniumTrioxide  8.940650e-07  1.226143e-06  6.141484e-05  
RockSalt

In [16]:
def position_MAE(y_pred, y_true):
    """
    Mean absolute error for positions
    """
    return torch.mean(
        torch.sqrt(
            torch.sum(
                torch.nn.functional.mse_loss(
                    y_pred, 
                    y_true, 
                    reduction='none'
                ),
            dim=1)),
        dim=0
    )
    
def calculate_cell_errors(pred_cell_params, gt_cell_params):
    """
    Calculate the cell errors
    """
    # Calculate side length error
    side_length_error = torch.nn.functional.l1_loss(
        torch.tensor(pred_cell_params[:3]),
        torch.tensor(gt_cell_params[:3]),
    ).item()

    # Calculate angle error
    angle_error = torch.nn.functional.l1_loss(
        torch.tensor(pred_cell_params[3:]),
        torch.tensor(gt_cell_params[3:]),
    ).item()

    return side_length_error, angle_error

def calculate_atom_accuracy(pred_atoms, gt_atoms):
    """
    Calculate the accuracy of the predicted atoms
    """
    gt_atoms_simple =  np.where(np.array(gt_atoms) == 8, 1, 2)
    
    return np.sum(np.array(pred_atoms) == gt_atoms_simple) / len(gt_atoms_simple) * 100

In [17]:
# TODO: Calculate positional error in Å
frac_positional_error_list = []
positional_error_list = []
cell_length_error_list = []
cell_angle_error_list = []
atom_accuracy_list = []

prior_frac_positional_error_list = []
prior_positional_error_list = []
prior_cell_length_error_list = []
prior_cell_angle_error_list = []
prior_atom_accuracy_list = []

for index in tqdm(range(len(df_combined))):
    row = df_combined.iloc[index]

    # De-normalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_params = np.array(row['true_cell_parameters'])
        cell_params = (cell_params * cell_stds.cpu().numpy()) + cell_means.cpu().numpy()
    else:
        cell_params = np.array(row['true_cell_parameters'])

    gt_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['true_cell_positions']),
        cell=cell_params,
        pbc=True,
    )
    gt_abs_positions = gt_unit_cell.get_positions()
    
    pred_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['pred_cell_positions']),
        cell=cell_params,
        pbc=True,
    )
    pred_abs_positions = pred_unit_cell.get_positions()
    
    prior_pred_unit_cell = Atoms(
        numbers=np.array(row['true_cell_atoms']),
        scaled_positions=np.array(row['pred_cell_positions_prior']),
        cell=cell_params,
        pbc=True,
    )
    prior_pred_abs_positions = prior_pred_unit_cell.get_positions()

    # Calculate positional error
    frac_positional_error = position_MAE(
        torch.tensor(row['pred_cell_positions']),
        torch.tensor(row['true_cell_positions']),
    ).numpy()
    frac_positional_error_list.append(frac_positional_error.item())
    
    prior_frac_positional_error = position_MAE(
        torch.tensor(row['pred_cell_positions_prior']),
        torch.tensor(row['true_cell_positions']),
    ).numpy()
    prior_frac_positional_error_list.append(prior_frac_positional_error.item())
    
    positional_error = position_MAE(
        torch.tensor(pred_abs_positions),
        torch.tensor(gt_abs_positions),
    ).numpy()
    positional_error_list.append(positional_error.item())
    
    prior_positional_error = position_MAE(
        torch.tensor(prior_pred_abs_positions),
        torch.tensor(gt_abs_positions),
    ).numpy()
    prior_positional_error_list.append(prior_positional_error.item())
    
    # Calculate cell length and angle error
    cell_length_error, cell_angle_error = calculate_cell_errors(
        row['pred_cell_parameters'],
        cell_params,
    )
    cell_length_error_list.append(cell_length_error)
    cell_angle_error_list.append(cell_angle_error)
    
    prior_cell_length_error, prior_cell_angle_error = calculate_cell_errors(
        row['pred_cell_parameters_prior'],
        cell_params,
    )
    prior_cell_length_error_list.append(prior_cell_length_error)
    prior_cell_angle_error_list.append(prior_cell_angle_error)
    
    # Calculate atom accuracy
    atom_accuracy = calculate_atom_accuracy(row['pred_cell_atoms'], row['true_cell_atoms'])
    atom_accuracy_list.append(atom_accuracy)
    
    prior_atom_accuracy = calculate_atom_accuracy(row['pred_cell_atoms_prior'], row['true_cell_atoms'])
    prior_atom_accuracy_list.append(prior_atom_accuracy)
    
# Add positional error to dataframe
df_combined['frac_positional_error (no unit)'] = frac_positional_error_list
df_combined['positional_error (Å)'] = positional_error_list
df_combined['cell_length_error (Å)'] = cell_length_error_list
df_combined['cell_angle_error (°)'] = cell_angle_error_list
df_combined['atom_accuracy'] = atom_accuracy_list

df_combined['prior_frac_positional_error (no unit)'] = prior_frac_positional_error_list
df_combined['prior_positional_error (Å)'] = prior_positional_error_list
df_combined['prior_cell_length_error (Å)'] = prior_cell_length_error_list
df_combined['prior_cell_angle_error (°)'] = prior_cell_angle_error_list
df_combined['prior_atom_accuracy'] = prior_atom_accuracy_list

  0%|          | 0/318 [00:00<?, ?it/s]

In [18]:
df_combined[['crystalType', 'cell_parameters', 'cell_atoms', 'cell_positions', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].groupby('crystalType').mean()

,cell_parameters,cell_atoms,cell_positions,frac_positional_error (no unit),positional_error (Å),cell_length_error (Å),cell_angle_error (°),atom_accuracy,prior_frac_positional_error (no unit),prior_positional_error (Å),prior_cell_length_error (Å),prior_cell_angle_error (°),prior_atom_accuracy
crystalType,,,,,,,,,,,,,
AntiFluorite,0.012128,4.494422e-06,0.000392,0.030124,0.165073,0.217165,1.588164,100.000000,0.038043,0.211324,0.206132,1.673850,100.000000
CadmiumChloride,0.005546,5.743260e-05,0.001353,0.046854,0.395938,0.193430,1.826515,100.000000,0.041858,0.353302,0.235527,1.851078,100.000000
CadmiumIodide,0.004281,9.773894e-04,0.003262,0.043659,0.165273,0.149154,0.690312,100.000000,0.026022,0.101131,0.136597,0.652890,100.000000
CaesiumChloride,0.004120,3.172901e-06,0.002523,0.065547,0.180471,0.154435,1.859435,100.000000,0.059387,0.163631,0.169735,1.831711,100.000000
Fluorite,0.006951,1.510689e-05,0.000505,0.028225,0.155286,0.180855,1.107034,100.000000,0.028003,0.155761,0.135107,1.128010,100.000000
NickelArsenide,0.007181,2.675642e-07,0.001181,0.041435,0.164808,0.142581,1.177106,100.000000,0.040157,0.161140,0.142962,1.193688,100.000000
RheniumTrioxide,0.000648,4.822641e-06,0.000352,0.028555,0.110743,0.056726,1.126688,100.000000,0.028515,0.110764,0.059059,1.097283,100.000000
RockSalt,0.014149,4.311671e-04,0.002777,0.053082,0.266606,0.239309,1.748313,100.000000,0.040883,0.184101,0.226844,1.648254,100.000000
Rutile,0.004107,4.257466e-08,0.003792,0.049109,0.213688,0.152139,2.659907,100.000000,0.048316,0.210180,0.149457,2.716094,100.000000


In [19]:
df_combined[['crystalType', 'cell_parameters', 'cell_atoms', 'cell_positions', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].groupby('crystalType').std()

,cell_parameters,cell_atoms,cell_positions,frac_positional_error (no unit),positional_error (Å),cell_length_error (Å),cell_angle_error (°),atom_accuracy,prior_frac_positional_error (no unit),prior_positional_error (Å),prior_cell_length_error (Å),prior_cell_angle_error (°),prior_atom_accuracy
crystalType,,,,,,,,,,,,,
AntiFluorite,0.018343,3.841417e-06,0.000148,0.005221,0.035436,0.201362,0.241607,0.00000,0.039442,0.222473,0.175931,0.486916,0.000000
CadmiumChloride,0.005974,2.762587e-04,0.003361,0.034005,0.290237,0.124447,0.247230,0.00000,0.014164,0.127745,0.173865,0.208839,0.000000
CadmiumIodide,0.006566,4.979068e-03,0.014610,0.076685,0.271867,0.123795,0.614066,0.00000,0.006185,0.036577,0.125191,0.386351,0.000000
CaesiumChloride,0.007744,1.582279e-05,0.004213,0.036048,0.100767,0.110772,0.582930,0.00000,0.006394,0.028121,0.101794,0.473063,0.000000
Fluorite,0.009789,7.262509e-05,0.001087,0.022385,0.132316,0.139463,0.410569,0.00000,0.021202,0.132198,0.116810,0.389476,0.000000
NickelArsenide,0.017523,6.978873e-07,0.002778,0.024407,0.095101,0.133257,0.163860,0.00000,0.013505,0.067259,0.113810,0.193062,0.000000
RheniumTrioxide,0.001024,1.186163e-05,0.000312,0.008992,0.034631,0.039900,0.141980,0.00000,0.010049,0.039422,0.042168,0.199627,0.000000
RockSalt,0.023988,1.251266e-03,0.006459,0.058931,0.349986,0.214475,0.441036,0.00000,0.043073,0.162077,0.218041,0.594894,0.000000
Rutile,0.005280,7.973623e-08,0.014557,0.025464,0.115490,0.084667,0.280763,0.00000,0.024227,0.110423,0.080546,0.233340,0.000000


In [20]:
df_combined[['cell_parameters', 'cell_atoms', 'cell_positions', 'kld', 'frac_positional_error (no unit)', 'positional_error (Å)', 'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy', 'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)', 'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)', 'prior_atom_accuracy']].describe().loc[['mean', 'std']].T

,mean,std
cell_parameters,0.006621,0.013927
cell_atoms,0.001304,0.021011
cell_positions,0.001882,0.008675
kld,0.015106,0.230335
frac_positional_error (no unit),0.042306,0.037497
positional_error (Å),0.212288,0.201885
cell_length_error (Å),0.164764,0.147189
cell_angle_error (°),1.671291,0.685355
atom_accuracy,99.977538,0.400552
prior_frac_positional_error (no unit),0.042164,0.055570


In [21]:
df_combined.columns

Index(['total', 'reconstruction_loss', 'cell_parameters', 'cell_positions',
       'cell_atoms', 'kld', 'particleSize', 'crystalType', 'n_atoms',
       'n_oxygens', 'n_metals', 'pred_cell_parameters', 'pred_cell_positions',
       'pred_cell_atoms', 'pred_cell_parameters_prior',
       'pred_cell_positions_prior', 'pred_cell_atoms_prior',
       'latent_space_mean', 'latent_space_std', 'latent_space_mean_prior',
       'latent_space_std_prior', 'true_cell_parameters', 'true_cell_positions',
       'true_cell_atoms', 'ls1', 'ls2', 'ls1_prior', 'ls2_prior',
       'frac_positional_error (no unit)', 'positional_error (Å)',
       'cell_length_error (Å)', 'cell_angle_error (°)', 'atom_accuracy',
       'prior_frac_positional_error (no unit)', 'prior_positional_error (Å)',
       'prior_cell_length_error (Å)', 'prior_cell_angle_error (°)',
       'prior_atom_accuracy'],
      dtype='object')

## Plot

In [22]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [23]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [24]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [25]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [26]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [27]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [28]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [29]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [30]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958046674728394, -1.4048123359680176]","[0.12149954587221146, 0.17361588776111603]","[0.2681826651096344, -1.3952175378799438]","[4.595865726470947, 4.592655658721924, 2.88950...","[[5.999999848427251e-05, -0.007619999814778566...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.349580,-1.404812
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034023851156235, 0.14894968271255493]","[-0.9407056570053101, 0.2841966152191162]","[8.731319427490234, 8.729973793029785, 8.71541...","[[0.508899986743927, 0.5084999799728394, 0.502...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.899346,0.109892
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640621185302734, -1.4373685121536255]","[0.11025752872228622, 0.1611802875995636]","[0.7794504165649414, -1.4691522121429443]","[3.5770926475524902, 3.4490699768066406, 2.547...","[[-0.09384000301361084, -0.06644999980926514, ...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.356406,-1.437369
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.4590970277786255, -0.8510302901268005]","[0.10611193627119064, 0.1737586259841919]","[-0.5558818578720093, -1.1748974323272705]","[5.163498401641846, 5.202596664428711, 5.09917...","[[-0.013380000367760658, -0.008259999565780163...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.459097,-0.851030
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.46544915437698364, -0.853926956653595]","[0.10532078146934509, 0.17308756709098816]","[-0.408798485994339, -0.6591929197311401]","[5.344516754150391, 5.352447986602783, 5.31303...","[[-0.0009800000116229057, 0.002349999966099858...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.465449,-0.853927


In [31]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [32]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [33]:
# Plot the experimental data
# index_list = [0,1,2] # Rutile samples
index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

# Interpolate in latent space

## Code

In [34]:
# Create latent samples for interpolation between two points
n_latent_samples = 7

# Define two latent points from pca space
# latent_point_1 = (24, 1) # ex1 (9, -13)
# latent_point_2 = (24, -15) # ex1 (24,-9)

# Define n latent points from latent space
crystal_types = ['NickelArsenide', 'CadmiumIodide'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']  ['NickelArsenide', 'CadmiumIodide', 'CadmiumChloride']
indices = [0, 4] #[14,3] #[0, 4, 0]  [0, 4, 3]
point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [35]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [36]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp['Interpolation number'] = df_interp['name'].apply(lambda x: int(x.split('_')[1]))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2,Interpolation number
0,sample_0,"[0.8023317456245422, 0.5319936871528625]","[3.105869770050049, 3.091283082962036, 5.26607...","[[0.004939999897032976, -0.015239999629557133,...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.802332,0.531994,0
1,sample_1,"[0.7796879410743713, 0.6852967739105225]","[2.939666748046875, 2.9236230850219727, 5.1730...","[[0.012439999729394913, -0.009050000458955765,...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.779688,0.685297,1
2,sample_2,"[0.7570441365242004, 0.8385998010635376]","[2.677729845046997, 2.640970230102539, 5.07746...","[[-0.00018000000272877514, -0.0202200002968311...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.757044,0.838600,2
3,sample_3,"[0.7344003915786743, 0.9919028282165527]","[2.3366951942443848, 2.324639081954956, 4.9764...","[[-0.009060000069439411, -0.03375000134110451,...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...",0.734400,0.991903,3
4,sample_4,"[0.7117565870285034, 1.1452059745788574]","[2.623650550842285, 2.6360554695129395, 4.4886...","[[-0.0028200000524520874, -0.02374999970197677...","[2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, ...",0.711757,1.145206,4


## Plot

In [37]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [38]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0, zorder=10, lw=2)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1, zorder=10, lw=2)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
line_1 = 3.5
line_2 = 4.5
line_3 = 8.5
line_4 = 10.5
ax0.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax0.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)
ax1.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [39]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'CadmiumIodide'
crystal_type_index = 0
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean_prior.iloc[dist_index]
dist_std = df_rec.latent_space_std_prior.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [40]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [41]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,Dist. mean,"[0.6624875068664551, 1.4868035316467285]","[3.0053775310516357, 3.0091404914855957, 4.304...","[[-0.0008200000156648457, -0.00170000002253800...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.662488,1.486804
1,sample_1,"[0.7576438188552856, 1.3422125577926636]","[2.9635138511657715, 2.9586915969848633, 4.229...","[[-0.0010900000343099236, -0.00319999991916120...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.757644,1.342213
2,sample_2,"[0.5489399433135986, 1.621418833732605]","[3.1506693363189697, 3.1687092781066895, 4.492...","[[-0.003920000046491623, -0.001019999966956675...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.548940,1.621419
3,sample_3,"[0.6508345603942871, 1.4268991947174072]","[2.971236228942871, 2.9805550575256348, 4.2405...","[[0.0013599999947473407, -0.0035500000230968, ...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.650835,1.426899
4,sample_4,"[0.601003110408783, 1.2780061960220337]","[2.922844171524048, 2.934497356414795, 4.14186...","[[0.007350000087171793, -0.0051299999468028545...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...",0.601003,1.278006


## Plot

In [42]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sampling_samples.pdf', dpi=300)

# Sample every latent distrubution and calculate loss

In [45]:
# Number of samples per point
n_samples = 100 #1000
# test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    cell_positions_padded = np.zeros((setup_json['model']['out_dim'], 3))
    cell_positions_padded[:len(df_rec['pred_cell_positions'].iloc[data_index]), :] = np.array(df_rec['pred_cell_positions'].iloc[data_index])
    # if setup_json['training']['simplified_atom_identities']:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 3))
    # else:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 119))
    cell_atoms_padded = np.zeros((setup_json['model']['out_dim'],))
    cell_atoms_padded[:len(df_rec['pred_cell_atoms'].iloc[data_index])] = np.array(df_rec['pred_cell_atoms'].iloc[data_index])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['pred_cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([cell_positions_padded]*(n_samples+1))
    ground_truth_cell_atoms.extend([cell_atoms_padded]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters, dtype=torch.float32)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions, dtype=torch.float32)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms, dtype=torch.long)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [46]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [47]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for batch_index, sample_index in enumerate(sample_indeces):
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        if data_crystal_types[sample_index] == 'Interpolated':
            sample_results['crystalType'].append(data_crystal_types[sample_index])#+ ' ' + str(sum(cell_atoms_true[batch_index] > 0).item()))
        else:
            sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [48]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.3365916907787323, -1.465637445449829]","[4.568840503692627, 4.567602157592773, 2.87808...","[[0.001500000013038516, -0.013120000250637531,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.101316,0.336592,-1.465637
1,0_0,sample,Rutile,"[0.5005266070365906, -1.2787153720855713]","[4.522429943084717, 4.500632286071777, 2.89023...","[[0.005969999823719263, -0.012199999764561653,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",1.227924,0.500527,-1.278715
2,0_1,sample,Rutile,"[0.41322141885757446, -1.7302595376968384]","[4.546882152557373, 4.535094738006592, 2.86739...","[[0.00925000011920929, -0.019009999930858612, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.429808,0.413221,-1.730260
3,0_2,sample,Rutile,"[0.39430904388427734, -1.6207951307296753]","[4.550436973571777, 4.544576644897461, 2.86057...","[[0.00887999963015318, -0.021570000797510147, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.186982,0.394309,-1.620795
4,0_3,sample,Rutile,"[0.3329276740550995, -1.6673121452331543]","[4.555886745452881, 4.553408622741699, 2.85623...","[[0.003000000026077032, -0.018319999799132347,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.138573,0.332928,-1.667312


In [49]:
# Plot sampling results in latent space
if 'Interpolation' in model_path:
    custom_palette = ['k', sns.color_palette('tab20')[4], sns.color_palette('tab20')[7]]
    custom_markers = ['o', (4,0,0), (4,1,45)]
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & (df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette[1:], markers=custom_markers[1:])
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & ~(df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', marker='o', s=50, palette='viridis')
        
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
else:
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.svg', dpi=300, transparent=True)

In [50]:
# Plot interpolation results over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.svg', dpi=300, transparent=True)

In [51]:
df_exp_results_2 = df_exp_results.copy()
df_exp_results_2['Label'] = ''
for i in range(len(df_exp_results_2)):
    if df_exp_results_2['composition'].iloc[i] == 'IrO2':
        df_exp_results_2['Label'].iloc[i] = 'IrO2 (Rutile)'
    elif df_exp_results_2['composition'].iloc[i] == 'CeO2':
        df_exp_results_2['Label'].iloc[i] = 'CeO2 (Fluorite)'
    else:
        df_exp_results_2['Label'].iloc[i] = 'HEA (Spinel)'
        
# # Remove index 1
# df_exp_results_2.drop(index=1, inplace=True)

In [52]:
# Plot experimental data over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='ls1', y='ls2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='pc1', y='pc2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_sample_density.pdf', dpi=300)

In [53]:
df_exp_results

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2,ground_truth_crystal_type
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.34958046674728394, -1.4048123359680176]","[0.12149954587221146, 0.17361588776111603]","[0.2681826651096344, -1.3952175378799438]","[4.595865726470947, 4.592655658721924, 2.88950...","[[5.999999848427251e-05, -0.007619999814778566...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.349580,-1.404812,Rutile
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-0.8993464112281799, 0.10989199578762054]","[0.07034023851156235, 0.14894968271255493]","[-0.9407056570053101, 0.2841966152191162]","[8.731319427490234, 8.729973793029785, 8.71541...","[[0.508899986743927, 0.5084999799728394, 0.502...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.899346,0.109892,Rutile
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.35640621185302734, -1.4373685121536255]","[0.11025752872228622, 0.1611802875995636]","[0.7794504165649414, -1.4691522121429443]","[3.5770926475524902, 3.4490699768066406, 2.547...","[[-0.09384000301361084, -0.06644999980926514, ...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.356406,-1.437369,Rutile
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.4590970277786255, -0.8510302901268005]","[0.10611193627119064, 0.1737586259841919]","[-0.5558818578720093, -1.1748974323272705]","[5.163498401641846, 5.202596664428711, 5.09917...","[[-0.013380000367760658, -0.008259999565780163...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.459097,-0.851030,Fluorite
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.46544915437698364, -0.853926956653595]","[0.10532078146934509, 0.17308756709098816]","[-0.408798485994339, -0.6591929197311401]","[5.344516754150391, 5.352447986602783, 5.31303...","[[-0.0009800000116229057, 0.002349999966099858...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.465449,-0.853927,Fluorite
5,CeO2,"[[0.0], [0.0007327766506932676], [0.0014382996...","[-0.4588150978088379, -0.8527802228927612]","[0.10431147366762161, 0.1725536286830902]","[-0.3936224579811096, -1.126720666885376]","[5.207348823547363, 5.2244391441345215, 5.1322...","[[-0.00698000006377697, -0.017920000478625298,...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",-0.458815,-0.852780,Fluorite
6,Cu0.5Ni0.5Cr0.66Mn0.66Fe0.66O4,"[[0.0], [-1.6920250345719978e-05], [-3.3054449...","[-0.7703374028205872, 0.14266054332256317]","[0.056033581495285034, 0.1240103617310524]","[-0.7248309254646301, 0.25244247913360596]","[8.119363784790039, 8.113112449645996, 8.08894...","[[0.523829996585846, 0.5229799747467041, 0.509...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.770337,0.142661,Spinel
7,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [0.001129354815930128], [0.00220868061...","[-0.7724447250366211, 0.15924935042858124]","[0.05540243908762932, 0.1225060224533081]","[-0.7920407056808472, 0.10563535988330841]","[8.38882064819336, 8.383069038391113, 8.340620...","[[0.5131700038909912, 0.5120599865913391, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.772445,0.159249,Spinel
8,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [5.6757980928523466e-05], [0.000110909...","[-0.7670082449913025, 0.16185835003852844]","[0.05355964973568916, 0.11947672814130783]","[-0.7833676338195801, 0.062257230281829834]","[8.371277809143066, 8.366296768188477, 8.31520...","[[0.5117899775505066, 0.5094900131225586, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.767008,0.161858,Spinel
9,Co0.33Cu0.33Ni0.33CrMnO4,"[[0.0], [-1.0808051229105331e-05], [-2.1100040...","[-0.7618623375892639, 0.14131325483322144]","[0.05590708181262016, 0.12357873469591141]","[-0.8498676419258118, 0.011254996061325073]","[8.543462753295898, 8.538898468017578, 8.48266...","[[0.5141500234603882, 0.5128600001335144, 0.50...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-0.761862,0.141313,Spinel


In [54]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.3365916907787323, -1.465637445449829]","[4.568840503692627, 4.567602157592773, 2.87808...","[[0.001500000013038516, -0.013120000250637531,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.101316,0.336592,-1.465637
1,0_0,sample,Rutile,"[0.5005266070365906, -1.2787153720855713]","[4.522429943084717, 4.500632286071777, 2.89023...","[[0.005969999823719263, -0.012199999764561653,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",1.227924,0.500527,-1.278715
2,0_1,sample,Rutile,"[0.41322141885757446, -1.7302595376968384]","[4.546882152557373, 4.535094738006592, 2.86739...","[[0.00925000011920929, -0.019009999930858612, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.429808,0.413221,-1.730260
3,0_2,sample,Rutile,"[0.39430904388427734, -1.6207951307296753]","[4.550436973571777, 4.544576644897461, 2.86057...","[[0.00887999963015318, -0.021570000797510147, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.186982,0.394309,-1.620795
4,0_3,sample,Rutile,"[0.3329276740550995, -1.6673121452331543]","[4.555886745452881, 4.553408622741699, 2.85623...","[[0.003000000026077032, -0.018319999799132347,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.138573,0.332928,-1.667312


In [55]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)#, hue_order=legend_order, style_order=legend_order)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', hue_order=legend_order, style_order=legend_order)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [56]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Rutile,"[0.3365916907787323, -1.465637445449829]","[4.568840503692627, 4.567602157592773, 2.87808...","[[0.001500000013038516, -0.013120000250637531,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.101316,0.336592,-1.465637
1,0_0,sample,Rutile,"[0.5005266070365906, -1.2787153720855713]","[4.522429943084717, 4.500632286071777, 2.89023...","[[0.005969999823719263, -0.012199999764561653,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",1.227924,0.500527,-1.278715
2,0_1,sample,Rutile,"[0.41322141885757446, -1.7302595376968384]","[4.546882152557373, 4.535094738006592, 2.86739...","[[0.00925000011920929, -0.019009999930858612, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.429808,0.413221,-1.730260
3,0_2,sample,Rutile,"[0.39430904388427734, -1.6207951307296753]","[4.550436973571777, 4.544576644897461, 2.86057...","[[0.00887999963015318, -0.021570000797510147, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.186982,0.394309,-1.620795
4,0_3,sample,Rutile,"[0.3329276740550995, -1.6673121452331543]","[4.555886745452881, 4.553408622741699, 2.85623...","[[0.003000000026077032, -0.018319999799132347,...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.138573,0.332928,-1.667312


In [57]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()

# Interpolation dataset input to model

In [74]:
# Load CHILI dataset
dataset_interp = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation_v2',
    graph_type=setup_json['data']['graph_type'],
)

# Load/create data splits
try:
    # Load existing data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
except FileNotFoundError:
    # Create new data split
    dataset_interp.create_data_split(
        test_size=setup_json['data']['split']['test'],
        validation_size=setup_json['data']['split']['validation'],
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
    # Load data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
# Dataloader
# data_loader_interp = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
data_loader_interp = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [78]:
# Load CHILI dataset
dataset_interp_uc = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation_v2',
    graph_type='unit_cell',
)

# Load existing data split
dataset_interp_uc.load_data_split(
    split_strategy=setup_json['data']['split_strategy'],
    stratify_on=setup_json['data']['stratify_column'],
)
    
# Dataloader
# dataset_interp_uc = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# dataset_interp_uc = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
dataset_interp_uc = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [79]:
interp_data_results = {
    'name': [],
    'crystal_type': [],
    'pdf': [],
    'n_atoms': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_atoms': [],
    'cell_positions': [],
}
    

In [81]:
# Inference
model.eval()
for batch in tqdm(dataset_interp_uc, desc='Inference', disable=setup_json['disable_tqdm']):
    batch = batch.to(device)
    this_batch_size = batch.batch.amax().item() + 1
    
    try:
        n_atoms = batch.y['unit_cell_n_atoms']
    except:
        n_atoms = batch.y['n_atoms']
    
    # Normalize scattering
    pdf = batch.y['xPDF'][:,1,:].unsqueeze(-1)
    if setup_json['data']['normalize_scattering']:
        # Normalize so highest peak in each sample is 1
        pdf /= torch.amax(pdf, dim=1, keepdim=True)[0]
    
    # Composition conditioning
    composition = torch.zeros(this_batch_size, 119).to(device)
    elements_in_batch = batch.y['atomic_species'].long()
    index_counter = 0
    for i in range(this_batch_size):
        n_elements = batch.y['n_atomic_species'][i]
        composition[i, elements_in_batch[index_counter:index_counter + n_elements]] = 1
        index_counter += n_elements
    composition[:, 0] = 1 
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
        
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store names
    interp_data_results['name'].extend(batch.data_id)
    interp_data_results['crystal_type'].extend(batch.y['crystal_type'])
    
    # Store PDF
    interp_data_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store number of atoms
    interp_data_results['n_atoms'].extend(n_atoms.cpu().tolist())
    
    # Store latent representation
    interp_data_results['prior_mean'].extend(prior_mean.cpu().tolist())
    interp_data_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    interp_data_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    interp_data_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_data_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_data_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # # Create CIF files
    # for batch_index in range(this_batch_size):
    #     # Prediction
    #     try:
    #         create_cif(
    #             cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
    #             cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
    #             cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
    #             filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
    #             prediction=True,
    #             composition=data_composition_string[composition_string_index[batch_index]],
    #             simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
    #         )
    #     except:
    #         print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_interp_data_results = pd.DataFrame(interp_data_results)
if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    df_interp_data_results[['ls1', 'ls2']] = df_interp_data_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_interp_data_results[['pc1', 'pc2']] = pca.transform(np.array(df_interp_data_results['prior_mean'].values.tolist()))
    
df_interp_data_results['Step'] = 0
df_interp_data_results['Step'][df_interp_data_results['crystal_type'] == 'interpolated'] = df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated']['name'].str.split('_').str[-3].str[-1].astype(int)

df_interp_data_results.head()

,name,crystal_type,pdf,n_atoms,prior_mean,prior_log_std,z_sample,cell_parameters,cell_atoms,cell_positions,ls1,ls2,Step
0,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.000650433823466301], [-0.001385843...",27,"[0.5236246585845947, -1.6123684644699097]","[0.056431304663419724, 0.11883806437253952]","[0.5049564838409424, -1.493993878364563]","[4.520420074462891, 4.509079933166504, 2.85854...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[[0.017270000651478767, -0.005499999970197678,...",0.523625,-1.612368,3
1,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.00041956984205171466], [-0.0008662...",27,"[0.3216722309589386, 1.8383911848068237]","[0.11844587326049805, 0.1398300677537918]","[0.17362160980701447, 2.083676338195801]","[3.5125274658203125, 3.562729835510254, 16.439...","[2, 1, 1, 1, 2, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, ...","[[-0.028610000386834145, -0.015490000136196613...",0.321672,1.838391,3
2,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [0.00019362939929123968], [0.000212970...",28,"[0.4139220714569092, -1.583296775817871]","[0.060027170926332474, 0.09942282736301422]","[0.3751828074455261, -1.499564290046692]","[4.557843208312988, 4.5560526847839355, 2.8683...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[[0.006560000125318766, -0.018559999763965607,...",0.413922,-1.583297,3
3,NickelArsenide_V,NickelArsenide,"[[0.0], [-0.0011531105265021324], [-0.00234128...",32,"[0.8094152212142944, 0.6322392225265503]","[0.0653667077422142, 0.07735161483287811]","[0.7609270811080933, 0.6096312999725342]","[3.013033628463745, 2.995323657989502, 5.20975...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.008050000295042992, -0.012520000338554382,...",0.809415,0.632239,0
4,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0004505524702835828], [-0.00092623...",27,"[0.8206459879875183, 0.24802753329277039]","[0.06555262207984924, 0.07961246371269226]","[0.7626084089279175, 0.36880794167518616]","[3.5479822158813477, 3.5186238288879395, 5.412...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.002369999885559082, -0.0005799999926239252...",0.820646,0.248028,3


## Plot

In [82]:
# Plot interpolation points in latent space

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='ls1', y='ls2', color='black', marker='o', s=50)
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.scatterplot(data=df_interp_data_results, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset.pdf', dpi=300)


In [83]:
# Plot interpolation points in latent space (# of atoms)

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)
    # sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)

    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='n_atoms', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='# of atoms')
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, hue_order=legend_order)

    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset_nAtoms.pdf', dpi=300)